In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import numpy as np

In [ ]:
path = 'token_data.tsv'
df = pd.read_csv(path, sep='\t')

In [ ]:
len(df), df.columns.tolist()

In [ ]:
df['group_str'] = df.apply(lambda row: f"LEMMA={row['lemma']}_POS={row['pos']}_PARTS_LEMMA={row['parts_lemma']}_PARTS_POS={row['parts_pos']}", axis=1)
df['group_str'].value_counts()

In [ ]:
sentence_df = pd.read_csv('sentence_dataframe.tsv', sep='\t')
len(sentence_df), sentence_df.columns

In [ ]:
sentence_df.set_index('hash', inplace=True, drop=True)

In [ ]:
group_pct_dict = {}
group_hash_to_str = {}
group_hashes = list(df['group_hash'].unique())
for group_hash in tqdm(group_hashes, total=len(group_hashes)):
    row = df[df['group_hash'] == group_hash].iloc[0]
    group_pct_dict[group_hash] = row['group_pct']
    group_hash_to_str[group_hash] = row['group_str']

In [ ]:
all_groups = list(df['group_hash'].unique())
all_groups = sorted(all_groups, key=lambda x: group_pct_dict[x], reverse=True)
[group_hash_to_str[hash] for hash in all_groups[:10]]

In [ ]:
# # sanity check: we already have these dataframes saved for the first 200 groups
# top_200_groups = all_groups[:200]
# first_200_df = df[df['group_hash'].isin(top_200_groups)].copy()
# first_200_df = first_200_df[~first_200_df['pos'].isin(['X', 'NUM', 'PUNCT'])]
# for i, row in first_200_df.iterrows():
#     path = f"dataframes/{row['pos']}/{row['lemma']}.tsv"
#     if not os.path.exists(path):
#         print(i, path)
#     assert(os.path.exists(path))

In [ ]:
# sanity check: we already have these dataframes saved for the first 400 groups
top_400_groups = all_groups[:400]
first_400_df = df[df['group_hash'].isin(top_400_groups)].copy()
first_400_df = first_400_df[~first_400_df['pos'].isin(['X', 'NUM', 'PUNCT'])]
for i, row in first_400_df.iterrows():
    path = f"dataframes/{row['pos']}/{row['lemma']}.tsv"
    if not os.path.exists(path):
        print(i, path)
    assert(os.path.exists(path))

In [ ]:
START = 400
END = 1000
top_groups = all_groups[START:END]
reduced_df = df[df['group_hash'].isin(top_groups)]
len(reduced_df)

In [ ]:
reduced_df.columns

In [ ]:
save_dir = 'dataframes'

In [ ]:
REQUIRED_COLUMNS = [
    'text',	
    'pronunciation',	
    'lemma',	
    'pos',	
    'translation_en',	
    'token_count',	
    'token_pct',	
    'group_count',	
    'group_pct',	
    'parts_lemma',	
    'parts_pos',	
    'xpos',	
    'deprel',	
    'feats',	
    'token_hash',	
    'group_hash',	
    'sentence_1_it',	
    'sentence_1_en',
    'sentence_2_it',
    'sentence_2_en',
    'sentence_3_it',
    'sentence_3_en',
]
group_hashes = list(reduced_df['group_hash'].unique())
for group_hash in tqdm(group_hashes, total=len(group_hashes)):
    group_df = reduced_df[reduced_df['group_hash'] == group_hash].copy()
    if group_df['lemma'].nunique() != 1:
        print(len(group_df))
        print(group_df['group_str'].unique())
        print(group_df['group_hash'].unique())
    if group_df['pos'].nunique() != 1:
        print(len(group_df))
        print(group_df['group_str'].unique())
        print(group_df['group_hash'].unique())
    assert(group_df['lemma'].nunique() == 1)
    assert(group_df['pos'].nunique() == 1)
    pos = group_df.iloc[0]['pos']
    lemma = group_df.iloc[0]['lemma']

    for i, row in group_df.iterrows():
        sentence_hashes = eval(row['sentences'])
        assert(type(sentence_hashes) is list)
        for j in range(3):
            if len(sentence_hashes) > j:
                sentence_hash = sentence_hashes[j]
                sentence_it = sentence_df.loc[sentence_hash, 'text_it']
                sentence_en = sentence_df.loc[sentence_hash, 'text_en']
                group_df.loc[i, f'sentence_{j+1}_it'] = sentence_it
                group_df.loc[i, f'sentence_{j+1}_en'] = sentence_en
    group_df['translation_en'] = ''
    group_df['pronunciation'] = ''
    group_df.drop(columns=['sentences'], inplace=True)
    save_path = os.path.join(save_dir, pos, f"{lemma}.tsv")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    cols = [
        'text', 'pronunciation', 'lemma', 'pos', 'translation_en', 'token_count', 'token_pct', 'group_count',
        'group_pct', 'parts_lemma', 'parts_pos', 'xpos', 'deprel', 'feats', 'token_hash', 'group_hash',
        'group_str', 'sentence_1_it', 'sentence_1_en', 'sentence_2_it',
        'sentence_2_en', 'sentence_3_it', 'sentence_3_en'
    ]
    assert(len(set(group_df.columns).difference(set(cols))) == 0)
    missing_cols = list(set(cols).difference(set(group_df.columns)))
    for missing_col in missing_cols:
        assert('sentence_' in missing_col)
        group_df[missing_col] = ''
    assert(set(cols) == set(group_df.columns))
    group_df = group_df[cols]
    for i, row in group_df.iterrows():
        for col in cols:
            if type(row[col]) == str:
                group_df.loc[i, col] = row[col].strip()

    group_df.drop(columns=['group_str'], inplace=True)
    # print(set(group_df.columns) - set(REQUIRED_COLUMNS))
    # print(set(REQUIRED_COLUMNS) - set(group_df.columns))
    assert(set(REQUIRED_COLUMNS) == set(group_df.columns))
    group_df.to_csv(save_path, sep='\t', index=False)